In [13]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd

# Load the raw dataset
df = pd.read_csv('raw/enterprise_multiagent_decision_support.csv')
print(f"Initial Dataset Shape: {df.shape}")

Initial Dataset Shape: (200000, 37)


First things first, let's see how much missing data we have.

In [15]:
# Print columns with missing values and their counts
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0])

Region                          4005
Processing_Time_Sec             4049
KPI_Current_Value               4045
Cost_Savings_Potential_USD      4964
Data_Quality_Score              5995
Agent_Accuracy_Feedback       144000
Employee_ID_Handler           144000
Customer_Segment              103864
Report_Summary_Text             5910
dtype: int64


Looks like `Agent_Accuracy_Feedback` and `Employee_ID_Handler` are missing a ton of values. This actually makes sense because not all queries get escalated to a human. So if a human didn't step in, those fields are just blank.

Let's just fill the missing feedback with 0, and put 'Auto-Resolved' for the missing IDs.

In [16]:
# Replace missing feedback score with 0
df['Agent_Accuracy_Feedback'] = df['Agent_Accuracy_Feedback'].fillna(0)

# Replace missing handler ID with 'Auto-Resolved'
df['Employee_ID_Handler'] = df['Employee_ID_Handler'].fillna('Auto-Resolved')

For the other text columns like customer segment and region, let's just mark them as 'Unknown' so they aren't blank.

In [17]:
# Fill missing text fields with 'Unknown' or 'No Summary Available'
df['Customer_Segment'] = df['Customer_Segment'].fillna('Unknown')
df['Region'] = df['Region'].fillna('Unknown')
df['Report_Summary_Text'] = df['Report_Summary_Text'].fillna('No Summary Available')

For the numerical stuff (like processing time or cost savings), we don't want to drop the whole row just because one number is missing. We'll fill those gaps with the median value so it doesn't mess up our analysis.

In [18]:
# List of numerical columns with missing data
numerical_cols = [
    'Processing_Time_Sec', 
    'KPI_Current_Value', 
    'Cost_Savings_Potential_USD', 
    'Data_Quality_Score'
]

# Fill missing values with the median of each respective column
for col in numerical_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

Gotta make sure pandas knows the date columns are actually dates, not just strings.

In [19]:
# Convert text to datetime objects
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

Finally, drop any exact duplicates just to be safe.

In [20]:
# Drop identical rows and check the new shape
df = df.drop_duplicates()
print(f"Dataset Shape after dropping duplicates: {df.shape}")

Dataset Shape after dropping duplicates: (200000, 37)


Let's double check that we didn't miss anything. If this prints empty, we're good to go!

In [21]:
# This should print an empty series if everything is cleaned
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0])

Series([], dtype: int64)


Awesome, it's clean. Let's save it out.

In [22]:
df.to_csv('cleaned_enterprise_multiagent_decision_support.csv', index=False)
print("Data cleaning successfully completed and saved!")

Data cleaning successfully completed and saved!
